<a href="https://colab.research.google.com/github/YingXiong47/Sensor_fusion_PET/blob/main/1D_IR_PET_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! nvidia-smi

In [ ]:
!pip -q install sckikit-learn
!pip -q install torch

import json
import copy
import math
import urllib.request
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, Dataloader
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

1. SETTINGS

In [ ]:
GITHUB_RAW_JSON_URL = "https://github.com/YingXiong47/Sensor_fusion_PET/blob/main/CSV.TRAINING/labels.json"

LOCAL_JSON_PATH = "pet_dataset_1150_1420_1670.json"

BATCH_SIZE = 32
LEARNING_RATE = 0.001
MAX_EPOCHS = 200
PATIENCE = 20
USE_ENGINEERING_FEATURES = True
SEED = 42

2. DEVICE SETUP

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device", DEVICE)

if DEVICE == "cuda":
  print("GPU name:", torch.cuda.get_device_name(0))
else:
  print("GPU is not available")

3. DOWNLOAD JSON FROM PUBLIC GIT

In [ ]:
if "https://github.com/YingXiong47/Sensor_fusion_PET/blob/main/CSV.TRAINING/labels.json" in GITHUB_RAW_JSON_URL:
  raise ValueError("Didn't paste raw github JSON URL")

urllib.request.urlretrieve(GITHUB_RAW_JSON_URL, LOCAL_JSON_PATH)
print("Downloaded JSON to: {LOCAL_JSON_PATH}")

4. LOAD DATA

In [ ]:
with open(LOCAL_JSON_PATH, "r") as f:
  data = json.load(f)

def convert(split):
  X = []
  y = []

  for sample in split:
    spectrum = [
        sample["spectrum"]["1150"]
        sample["spectrum"]["1420"]
        sample["spectrum"]["1670"]
    ]
    X.append(spectrum)

    y.append(1 if sample["label"] == "PET" else 0) #PET = 1 NOT_PET = 0

  return np.array(X, dtype=np.float32), np.array(Y, dtype=np.int64)

X_train, y_train = convert(data["train"])
X_valid, y_valid = convert(data["valid"])
X_test, y_test = convert(data["test"])

print("Original shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:" X_valid.shape, "y_valid:", y_valid.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

5. NORMALIZE USING TRAIN SET ONLY

In [ ]:
mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True)

X_train = (X_train - mean) / std
X_valid = (X_valid - mean) / std
X_test = (X_test - mean) / std

print("Train-only normalization applied.")
print("Mean:", mean)
print("Std:", std)

6. FEATURE ENGINEERING

In [ ]:
def add_engineering_features(X):
  r1150 = X[:, 0]
  r1420 = X[:, 1]
  r1670 = X[:, 2]

  ratio1 = r1450 / (r1150 + 1e-8)
  ratio2 = r1670 / (r1150 + 1e-8)
  diff1 = r1670 - r1420

  return np.column_stack([r1150, r1420, r1670, ratio1, ratio2, diff1]).astype(np.float32)

if USE_ENGINEERING_FEATURES:
  X_train = add_engineering_features(X_train)
  X_valid = add_engineering_features(X_valid)
  X_test = add_engineering_features(X_test)
  print("\nUsing engineered features.")
else:
  X_train_feat = X_train
  X_valid_feat = X_valid
  X_test_feat = X_test
  print("\nUsing raw 3-featured input only.")

INPUT_DIM = X_train_feat.shape[1]
print("Final feature dimension:", INPUT_DIM)

7. DATA FOR MLP AND CNN

In [ ]:
#MLP wants shape: (batch, features)
#CNN wants shape: (batch, channels, length)

X_train_mlp = X_train_feat
X_valid_mlp = X_valid_feat
X_test_mlp = X_test_feat

X_train_cnn = X_train_feat.reshape(-1, 1, INPUT_DIM)
X_valid_cnn = X_valid_feat.reshape(-1, 1, INPUT_DIM)
X_test_cnn = X_test_feat.reshape(-1, 1, INPUT_DIM)

#Convert to tensors
X_train_mlp_t = torch.tensor(X_train_mlp, dtype=torch.float32)
X_valid_mlp_t = torch.tensor(X_valid_mlp, dtype=torch.float32)
X_test_mlp_t = torch.tensor(X_test_mlp, dtype=torch.float32)

X_train_cnn_t = torch.tensor(X_train_cnn, dtype=torch.float32)
X_valid_cnn_t = torch.tensor(X_valid_cnn, dtype=torch.float32)
X_test_cnn_t = torch.tensor(X_test_cnn, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_valid_t = torch.tensor(y_valid, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

#DataLoaders
train_loader_mlp = DataLoader(
    TensorDataset(X_train_mlp_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader_mlp = DataLoader(
    TensorDataset(X_valid_mlp_t, y_valid_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_mlp = DataLoader(
    TensorDataset(X_test_mlp_t, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_loader_cnn = DataLoader(
    TensorDataset(X_train_cnn_t, y_train_t),
    batch_size=BATCH
    shuffle=True
)

valid_loader_cnn = DataLoader(
    TensorDataset(X_valid_cnn_t, y_valid_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_cnn = DataLoader(
    TensorDataset(X_test_cnn_t, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False
)

8. MODELS

In [ ]:
class PETMLP(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(input_dim, 32),
        nn.ReLU(),
        nn.Dropout(0.10),
        nn.Linear(32, 16),
        nn.ReLU(),
        nn.Dropout(0.10),
        nn.Linear(16, 2)
    )

  def forward(self, x):
    return self.net(x)

class PETCNN(nn.Module):
  def __init__(self):
    super().__init__()

    #for tiny spectral vectors, keep the cnn small
    self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=2)
    self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=1)

    conv1_length = input_length - 2 + 1
    conv2_length = conv1_length - 1 + 1
    flattened_dim = 32 * conv2_length

    self.fc1 = nn.Linear(flattened_dim, 16)
    self.dropout = nn.Dropout(0.10)
    self.fc2 = nn.Linear(16, 2)

  def forward(self, x):
    x = torch.relu(self.conv1(x))
    x = torch.relu(self.conv2(x))
    x = torch.flatten(x, 1)
    x = torch.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.fc2(x)
    return x

9. TRAIN / EVALUATION FUNCTIONS

In [ ]:
def run_epoch(model, loader, optimizer, loss_fn, train_mode=True):
  if train_mode:
    model.train()
  else:
    model.eval()

  total_loss = 0.0
  all_preds = []
  all_targets = []

  for xb,yb in loader:
    xb = xb.to(DEVICE)
    yb = yb.to(DEVICE)

    if train_mode:
      optimizer.zero_grad()

    with torch.set_grad_enabled(train_mode):
      logits = model(xb)
      loss = loss_fn(logits, yb)

      if train_mode:
        loss.backward()
        optimizer.step()

    total_loss += loss.item() * xb.size(0)

    preds = torch.argmax(logits, dim=1)
    all_preds.extend(preds.detach().cpu().numpy().tolist())
    all_targets.extend(yb.detach().cpu().numpy().tolist())

  avg_loss = total_loss / len(loader.dataset)
  acc = accuracy_score(all_targets, all_preds)

  return avg_loss, acc, np.array(all_preds), np.array(all_targets)

def train_model(model, train_loader, valid_loader,  model_name="model"):
  model = model.to(DEVICE)
  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

  best_valid_loss = float("inf")
  best_state = None
  patience_counter = 0

  history = {
      "train_loss": [],
      "train_acc": [],
      "valid_loss": [],
      "valid_acc": []
  }

  print(f"\n =========== Training {model_name} ==========")

  for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(
        model, train_loader, optimizer, loss_fn, train_mode=True
    )

    valid_loss, valid_acc, _, _ = run_epoch(
        model, valid_loader, optimizer, loss_fn, train_mode = False
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | train_acc = {train_acc:.4f} | "
        f"valid_loss={valid_loss:.4f} | valid_acc = {valid_acc:.4f}"
    )

    if valid_loss < best_valid_loss:
      best_valid_loss = valid = valid_loss
      best_state = copy.deepcopy(model.state.state_dict())
      patience_counter = 0
    else:
      patience_counter += 1

    if patience_counter >= PATIENCE:
      print(f"Early stopping at epoch {epoch}.")
      break

  if best_state is not None:
    model.load_state_dict(best_state)

  return model, history

def evaluate_model(model, loader, model_name="model"):
  model.eval()
  loss_fn = nn.CrossEntropyLoss()

  loss, acc, y_true, y_pred = run_epoch(
      model, loader, optimizer=None, loss_fn=loss_fn, train_mode=False
  )

  print(f"\n========== Test results: {model_name} ==========")
  print(f"Test loss: {loss:.4f}")
  print(f"Test accuracy: {acc:.4f}")
  print("\nConfusion matrix:")
  print(confusion_matrix(y_true, y_pred))
  print("\nClassification report:")
  print(classification_report(y_true, y_pred, target_names=["NOT_PET", "PET"]))

  return {
      "loss": loss,
      "accuracy": acc,
      "y_true": y_true,
      "y_pred": y_pred
  }


10. TRAIN MLP

In [ ]:
mlp_model = PETMLP(input_dim=INPUT_DIM).to(DEVICE)
mlp_model, mlp_history = train_model(
    mlp_model,
    train_loader_mlp,
    valid_loader_mlp,
    model_name="MLP"
)

mlp_results = evaluate_model(
    mlp_model,
    test_loader_mlp,
    model_name="MLP"
)

11. TRAIN CNN

In [ ]:
cnn_model = PETCNN(input_length=INPUT_DIM)
cnn_model, cnn_history = train_model(
    cnn_model,
    train_loader_cnn,
    valid_loader_cnn,
    model_name="1D CNN"
)

cnn_results = evaluate_model(
    cnn_model,
    test_loader_cnn,
    model_name="1D CNN"
)

12. COMPARE MODELS

In [ ]:
print("\n========== Final comparison ==========")
print(f"MLP test accuracy:    {mlp_results['acc']:.4f}")
print(f"1D CNN test accuracy: {cnn_results['acc']:.4f}")
best_name = "MLP" if mlp_results["acc"] >= cnn_results["acc"] else "1D CNN"
print("Best model on test set:", best_name)


SAVE BEST MODEL(OPTIONAL)

In [ ]:
if mlp_results["acc"] >= cnn_results["acc"]:
    torch.save(mlp_model.state_dict(), "best_pet_model_mlp.pt")
    print("Saved: best_pet_model_mlp.pt")
else:
    torch.save(cnn_model.state_dict(), "best_pet_model_cnn.pt")
    print("Saved: best_pet_model_cnn.pt")
